### Задача

Работаем аналитиком в продуктовой команде, которая занимается оптимизацией
рекомендательной системы в агрегаторе доставки продуктов.
Был запущен ABC-тест, чтобы одновременно протестировать два новых воздействия (два
новых ML-алгоритма для рекомендательной системы).
В файле Ticket 1. Dataset.csv содержатся результаты данного теста.
Следуем следующему алгоритму:
1. Исследуем метрику **суммарный GMV на клиента**:
- ANOVA-тест для выявления наличия хотя бы одной отличающейся группы;

- Тест со множественной проверкой гипотез с целью валидации В воздействия, а также сплит-
системы;

2. Исследуем метрику **конверсия клиента во второй заказ**:

- Тест со множественной проверкой гипотез с целью валидации В воздействия, а также сплит-
системы;

3. Исследуем метрику **средний GMV на клиентский заказ** (*ratio-метрика*):

- Тест со множественной проверкой гипотез с целью валидации В воздействия, а также сплит-
системы;

In [9]:
import numpy as np
import pandas as pd
from scipy import stats

In [6]:
df_orders = pd.read_csv('Ticket 1. Dataset.csv')
df_orders.head(3)

,id_order,id_client,group,gmv
0,109001921,38,A,10738
1,109001265,128,C,9537
2,109001407,109,B,5991


### Исследуем метрику **суммарный GMV на клиента**

In [7]:
df = df_orders.groupby(['group', 'id_client'])['gmv'].sum().reset_index()
df.head(3)

,group,id_client,gmv
0,A,11,26842
1,A,12,16113
2,A,13,26769


Проведем ANOVA тест (межгрупповое расстояние / внутригрупповое)

In [10]:
groups = {group: gmv.values for group, gmv in df.groupby('group')['gmv']}
bss = sum([len(gmv)*(gmv.mean() - df['gmv'].mean())**2 for gmv in groups.values()])
wss = sum([sum((gmv - gmv.mean())**2) for gmv in groups.values()])

k = len(groups)
n = len(df)
F = (bss / (k - 1)) / (wss/ (n - k))

p_val = 1 - stats.f.cdf(F, (k-1), (n-k))
F, p_val

(np.float64(4.346779577702097), np.float64(0.013557931317095151))

In [ ]:
stats.f_oneway(*groups.values()) # проверка готовым инструментом

F_onewayResult(statistic=np.float64(4.346779577702096), pvalue=np.float64(0.013557931317095106))

Тест показал стат значимость различий в группах => эффект от фичи есть хоть где-то

Проведем погрупповые тесты с поправкой Холма на уровень значимости при множественном тесте

In [16]:
from scipy.stats import ttest_ind
print(ttest_ind(groups['C'], groups['B'], equal_var=True)) # 0,05 / 3 = 0.0166
print(ttest_ind(groups['A'], groups['C'], equal_var=True)) # 0,05 / 2 = 0.025
print(ttest_ind(groups['A'], groups['B'], equal_var=True)) # 0,05


TtestResult(statistic=np.float64(2.731885993616697), pvalue=np.float64(0.006701878568992417), df=np.float64(277.0))
TtestResult(statistic=np.float64(-2.0257695206652526), pvalue=np.float64(0.04378288532023111), df=np.float64(267.0))
TtestResult(statistic=np.float64(0.721865653613026), pvalue=np.float64(0.4710111524487248), df=np.float64(266.0))


Если говорить только о В, то он стат значимо отличается от С

### Теперь посмотрим на **конверсию во второй заказ**

In [17]:
df = df_orders.groupby(['id_client', 'group']).agg(conv=('id_order', lambda x: int(len(x) > 1))).reset_index()
groups = {group: (conv, np.ones_like(conv)) for group, conv in df.groupby('group')['conv']}

Конверсия это ratio метрика, поэтому стат значимость разницы в группах будем оценивать через Дельта-метод 

In [19]:
def delta_method(num_t, denom_t, num_c, denom_c):
    def sample_ratio_var(num, denom):
        mean_num, mean_denom = np.mean(num), np.mean(denom)
        var_num, var_denom = np.var(num), np.var(denom)
        cov = np.cov(num, denom)[0, 1]
        # main formula for estimating variance of ratio-metric
        ratio_var = (
            (var_num / mean_denom**2)
            - (2 * (mean_num / mean_denom**3) * cov)
            + ((mean_num**2 / mean_denom**4) * var_denom)
        )
        return ratio_var

    ratio_t, ratio_c = num_t.sum()/denom_t.sum(), num_c.sum()/denom_c.sum()
    var_t, var_c = sample_ratio_var(num_t, denom_t), sample_ratio_var(num_c, denom_c)
    n_t, n_c = len(num_t), len(num_c)

    uplift = ratio_t - ratio_c
    se = np.sqrt(var_t/n_t + var_c/n_c)

    t = uplift / se
    p_value = (1 - stats.norm.cdf(abs(t))) * 2
    return p_value

In [20]:
for c, t in [('A', 'B'), ('A', 'C'), ('B', 'C')]:
  num_c, denom_c = groups[c]
  num_t, denom_t = groups[t]
  print(c, t, ':', num_c.mean(), num_t.mean(), 'p_val=', delta_method(num_t, denom_t, num_c, denom_c))

A B : 0.8604651162790697 0.8705035971223022 p_val= 0.8099153995531427
A C : 0.8604651162790697 0.8142857142857143 p_val= 0.3031049728655981
B C : 0.8705035971223022 0.8142857142857143 p_val= 0.19610199656354332


Получается, что стат значимой разницы нет

### **Средний GMV на клиентский заказ**

Аналогично предыдущей метрике

In [26]:
df = df_orders.groupby(['id_client', 'group']).agg(conv=('gmv', lambda x: x.mean())).reset_index()
groups = {group: (conv, np.ones_like(conv)) for group, conv in df.groupby('group')['conv']}

In [28]:
for c, t in [('A', 'B'), ('A', 'C'), ('B', 'C')]:
  num_c, denom_c = groups[c]
  num_t, denom_t = groups[t]
  print(c, t, ':', num_c.mean(), num_t.mean(), 'p_val=', delta_method(num_t, denom_t, num_c, denom_c))

A B : 6660.877445551862 6249.07865707434 p_val= 0.11554627216954172
A C : 6660.877445551862 7630.380119047618 p_val= 0.0003105533186988918
B C : 6249.07865707434 7630.380119047618 p_val= 7.069176044538494e-08


Стат значима разница только в A C (0.0003 < 0.0166)